In [ ]:
from diffusers import StableDiffusionPipeline
import torch

model_id = "sd-legacy/stable-diffusion-v1-5"
pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16)
pipe = pipe.to("cuda")  # Move to GPU if available

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/342 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/806 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/492M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

In [ ]:
pipe

StableDiffusionPipeline {
  "_class_name": "StableDiffusionPipeline",
  "_diffusers_version": "0.32.2",
  "_name_or_path": "sd-legacy/stable-diffusion-v1-5",
  "feature_extractor": [
    "transformers",
    "CLIPImageProcessor"
  ],
  "image_encoder": [
    null,
    null
  ],
  "requires_safety_checker": true,
  "safety_checker": [
    "stable_diffusion",
    "StableDiffusionSafetyChecker"
  ],
  "scheduler": [
    "diffusers",
    "PNDMScheduler"
  ],
  "text_encoder": [
    "transformers",
    "CLIPTextModel"
  ],
  "tokenizer": [
    "transformers",
    "CLIPTokenizer"
  ],
  "unet": [
    "diffusers",
    "UNet2DConditionModel"
  ],
  "vae": [
    "diffusers",
    "AutoencoderKL"
  ]
}

In [ ]:
pipe.text_encoder

CLIPTextModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 768)
      (position_embedding): Embedding(77, 768)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPSdpaAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
            (fc2): Linear(in_features=3072, out_features=768, bias=True)
          )
          (layer_norm2): LayerNorm((768,), ep

In [ ]:
prompt = "a photograph of an astronaut riding a horse"

In [ ]:
text_inputs = pipe.tokenizer(
    prompt,
    padding="max_length",
    max_length=pipe.tokenizer.model_max_length,
    truncation=True,
    return_tensors="pt"
)

In [ ]:
input_ids = text_inputs.input_ids.to(pipe.text_encoder.device)

In [ ]:
# Get text embeddings (CLIP's output)
with torch.no_grad():
    text_embeddings = pipe.text_encoder(input_ids)[0]

# Output details
print("Text embeddings shape:", text_embeddings.shape)  # [batch_size, sequence_length, hidden_size]
print("\nFirst 5 elements of the first token's embedding:")
print(text_embeddings[0, 0, :5].cpu().numpy())  # Convert to numpy for readability

Text embeddings shape: torch.Size([1, 77, 768])

First 5 elements of the first token's embedding:
[-0.3884   0.02295 -0.05234 -0.1842  -0.02739]


In [ ]:
with torch.no_grad():
    text_embeddings = pipe.text_encoder(input_ids,
                                        output_hidden_states=True,
                                        return_dict=True)

In [ ]:
text_embeddings

BaseModelOutputWithPooling(last_hidden_state=tensor([[[-0.3884,  0.0229, -0.0523,  ..., -0.4902, -0.3066,  0.0674],
         [ 0.0297, -1.3242,  0.3076,  ..., -0.5234,  0.9746,  0.6650],
         [ 0.4587,  0.5635,  1.6680,  ..., -1.9502, -1.2295,  0.0098],
         ...,
         [-3.0430, -0.0680, -0.1783,  ...,  0.3955, -0.0186,  0.7676],
         [-3.0547, -0.1063, -0.1935,  ...,  0.4246, -0.0188,  0.7588],
         [-2.9863, -0.0850, -0.1708,  ...,  0.4355,  0.0086,  0.7480]]],
       device='cuda:0', dtype=torch.float16), pooler_output=tensor([[-2.6328e+00, -5.6543e-01, -5.6549e-02,  3.9136e-01, -1.8203e+00,
          1.3320e+00, -2.2305e+00,  1.2783e+00,  1.0996e+00,  2.4182e-01,
         -1.6418e-01,  3.7427e-01,  1.0223e-02,  8.3105e-01,  2.5879e-01,
          1.3760e+00, -1.0391e+00, -6.6553e-01,  3.9795e-01,  1.6270e+00,
          6.5247e-02,  9.5752e-01, -2.0566e+00,  1.4990e+00, -5.7520e-01,
         -3.6646e-01,  7.7588e-01, -1.9746e+00,  2.5977e-01,  1.9849e-01,
         

In [ ]:
len(text_embeddings.hidden_states)

13